# Task 2: Yelp Polarity sentiment classification (Sneha)

Everything for Task 2 is in this one notebook: settings, data download, the shared split, preprocessing and its analysis,
three models trained from scratch, every required metric, the McNemar tests, the stopword/stemming ablation, and the
20-error review candidates. Run it top to bottom.

**How to run**
1. Set `SMOKE = True` in the first cell and choose *Run All*. It uses a few thousand reviews and one epoch, takes a few minutes,
   and proves everything works. Its outputs go to `task2_sentiment/sneha/_smoke/`.
2. Set `SMOKE = False`, choose *Restart & Run All*, then **save the notebook with its outputs**.
3. If a lab session ends mid-way, set `RESUME = True` and *Run All*. Models already trained with the same settings are reused,
   not retrained.

**The three models**, a ladder of one idea each, every embedding learned from scratch (no pretrained embeddings or LMs):

| | Model | What it tests |
|---|---|---|
| M1 baseline | `meanpool`: averaged word embeddings + MLP | the floor; no word order at all (Iyyer et al. 2015) |
| M2 experimental | `bigru`: BiGRU, max-pooled over time | does word order buy anything over averaging? |
| M3 experimental | `charcnn`: character CNN (Zhang, Zhao and LeCun 2015) | can a model that never sees a word match one that does? |

Labels follow the dataset's readme: class 1 is negative and class 2 is positive. In this notebook, 1 → 0 and 2 → 1.

## 1. Settings
Every hyperparameter and path is in `task2_sentiment/sneha/config.yaml`, so the run is config-driven and nothing below hard-codes a value. Values marked SHARED there must match Ayush's settings. Set `SMOKE` and `RESUME` here.

In [ ]:
SMOKE  = False    # True: a few thousand reviews, 1 epoch; proves the notebook runs
RESUME = False    # True: reuse models already trained with exactly these settings

# Every hyperparameter and path lives in task2_sentiment/sneha/config.yaml: "config" for the real run,
# "smoke" for the overrides that SMOKE = True applies. smoke_test.py sets DATA266_SMOKE=1.
import os, pathlib, yaml
SMOKE = SMOKE or os.environ.get("DATA266_SMOKE") == "1"
_root = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
             if (p / "requirements.txt").is_file())
CONFIG_FILE = _root / "task2_sentiment" / "sneha" / "config.yaml"
_settings = yaml.safe_load(CONFIG_FILE.read_text(encoding="utf-8"))
CONFIG, SMOKE_OVERRIDES = _settings["config"], _settings["smoke"]
print(f"settings from {CONFIG_FILE.relative_to(_root).as_posix()} | SMOKE = {SMOKE} | RESUME = {RESUME}")

## 2. Setup

In [ ]:
import csv, hashlib, json, logging, math, os, platform, random, re, shutil, subprocess, sys, tarfile, tempfile, time
import urllib.request
from collections import Counter
from functools import lru_cache
from importlib import metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from IPython.display import Image, Markdown, display
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "requirements.txt").is_file())
os.chdir(ROOT)


class Cfg(dict):
    """The settings dict with attribute access: cfg.batch_size."""
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError as e:
            raise AttributeError(key) from e


cfg = Cfg({**CONFIG, **(SMOKE_OVERRIDES if SMOKE else {})})
CFG_HASH = hashlib.sha256(json.dumps(cfg, sort_keys=True, separators=(",", ":"),
                                     ensure_ascii=False).encode("utf-8")).hexdigest()


def rel(p) -> str:
    """Repo-relative path for logs and manifests, so no personal path is ever recorded."""
    p = Path(p).resolve()
    try:
        return p.relative_to(ROOT).as_posix()
    except ValueError:
        return p.as_posix()


OUT = ROOT / cfg.out_dir
CKPT_DIR, OUTPUTS, PROCESSED = OUT / "checkpoints", OUT / "outputs", OUT / "data_processed"
for d in (CKPT_DIR, OUTPUTS, PROCESSED, ROOT / cfg.logs_dir, ROOT / cfg.manifests_dir):
    d.mkdir(parents=True, exist_ok=True)
RAW = ROOT / cfg.raw_dir / "yelp_review_polarity_csv"
SPLITS = ROOT / cfg.splits_dir
METRICS_CSV, ABLATION_CSV = OUT / "metrics_report.csv", OUT / "ablation_report.csv"
ERRORS_MD = OUTPUTS / "error_review_candidates.md"
MANIFEST = ROOT / cfg.manifests_dir / (f"{cfg.member}_task2.json" if cfg.run_name == "main"
                                       else f"{cfg.member}_task2_{cfg.run_name}.json")
print(f"repo {ROOT.name} | {'SMOKE' if SMOKE else 'REAL'} run '{cfg.run_name}' | settings sha256 "
      f"{CFG_HASH[:12]} | outputs -> {rel(OUT)}")

## 3. Reproducibility helpers
This cell covers seeding, the GPU check, raw logs that can't be overwritten, and the manifest helpers.

In [ ]:
def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def arch_supported(cap: tuple[int, int], archs: list[str]) -> bool:
    """CUDA rules: an sm_XY binary runs on any sm_XZ with Z >= Y (same major version);
    compute_XY PTX can be JIT-compiled for any newer card. torch 2.4.1 serves the RTX 4090
    (sm_89) with its sm_86 binaries; an RTX 5090 (sm_120) needs torch >= 2.7."""
    major, minor = cap
    c = 10 * major + minor
    return not archs or any(
        (a.startswith("sm_") and int(a[3:]) // 10 == major and int(a[3:]) <= c)
        or (a.startswith("compute_") and int(a[8:]) <= c) for a in archs)


def get_device(requested: str) -> torch.device:
    """cuda: a supported GPU or an error, never a silent CPU fallback. auto: GPU if present. cpu: CPU."""
    if requested == "cpu":
        return torch.device("cpu")
    if requested not in ("cuda", "auto"):
        raise ValueError(f"device must be cuda, auto or cpu, not {requested!r}")
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(0)
        if not arch_supported(cap, torch.cuda.get_arch_list()):
            raise RuntimeError(f"{torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]}) is not "
                               f"supported by torch {torch.__version__}. Use an RTX 4090.")
        return torch.device("cuda")
    if requested == "auto":
        return torch.device("cpu")
    raise RuntimeError(f"device='cuda' but torch {torch.__version__} sees no CUDA GPU "
                       f"(torch.version.cuda = {torch.version.cuda}; None means a CPU-only "
                       f"wheel: reinstall with the README's CUDA command). For a deliberate "
                       f"CPU run set device='auto' or 'cpu'.")


def device_name(device: torch.device) -> str:
    return torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu"


def cpu_name() -> str:
    return platform.processor() or platform.machine()


def sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def reset_peak_mem() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def peak_mem_gb() -> float:
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def git_sha() -> str:
    """The checked-out commit. Reads .git directly when the git program is missing
    (the lab's Docker container has none)."""
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        pass
    try:
        g = ROOT / ".git"
        head = (g / "HEAD").read_text().strip()
        if not head.startswith("ref: "):
            return head                                   # detached HEAD: the hash itself
        ref = head[5:]
        if (g / ref).exists():
            return (g / ref).read_text().strip()
        packed = g / "packed-refs"
        for line in (packed.read_text().splitlines() if packed.exists() else []):
            if line.endswith(" " + ref):
                return line.split()[0]
    except Exception:
        pass
    return "unknown"


def installed_packages() -> list[str]:
    return sorted(f"{d.metadata['Name']}=={d.version}" for d in metadata.distributions())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


def open_run_log(stage: str) -> tuple[logging.Logger, Path]:
    """A new timestamped raw log, opened in mode "x" so an existing log can never be
    overwritten. Every line goes to the file; INFO lines also show in the notebook."""
    stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    path = ROOT / cfg.logs_dir / f"task2_{stage}_{cfg.run_name}_{stamp}.log"
    while path.exists():                                  # two stages in the same second
        time.sleep(1)
        stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
        path = ROOT / cfg.logs_dir / f"task2_{stage}_{cfg.run_name}_{stamp}.log"
    logger = logging.getLogger(f"task2.{stage}.{cfg.run_name}.{stamp}")
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(path, mode="x", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    sh = logging.StreamHandler(sys.stdout)
    sh.setLevel(logging.INFO)
    for h in (fh, sh):
        h.setFormatter(fmt)
        logger.addHandler(h)
    logger.info(f"config_sha256 {CFG_HASH}")           # first line, always
    logger.info(f"git_sha {git_sha()} | notebook task2_sentiment/sneha/src/task2_sentiment_sneha.ipynb")
    logger.info(f"device {DEVICE.type} | {device_name(DEVICE)} | cpu {cpu_name()}")
    logger.info(f"torch {torch.__version__} | cuda {torch.version.cuda} | python "
                f"{platform.python_version()} | platform {platform.platform()}")
    logger.debug("config " + json.dumps(cfg, sort_keys=True, ensure_ascii=False))
    logger.debug("packages " + json.dumps(installed_packages()))
    logger.info(f"log_path {rel(path)}")
    return logger, path


def close_run_log(logger: logging.Logger) -> None:
    for h in list(logger.handlers):
        h.close()
        logger.removeHandler(h)


def snapshot_config(log_path: Path) -> None:
    """config_used.yaml and logs.md in the task folder (brief checklist)."""
    (OUT / "config_used.yaml").write_text(
        f"# settings of the most recent run; sha256 {CFG_HASH}\n"
        + yaml.safe_dump(dict(cfg), sort_keys=False, allow_unicode=True), encoding="utf-8")
    logs_md = OUT / "logs.md"
    if not logs_md.exists():
        logs_md.write_text("# Raw logs for this task\n\nAppended automatically at the start of "
                           "every stage. The logs themselves are never edited.\n\n", encoding="utf-8")
    with open(logs_md, "a", encoding="utf-8") as f:
        f.write(f"- `{rel(log_path)}` (run `{cfg.run_name}`, settings sha256 `{CFG_HASH[:12]}`)\n")


set_seeds(cfg.seed)
DEVICE = get_device(cfg.device)
print(f"device: {device_name(DEVICE)} | cpu: {cpu_name()} | torch {torch.__version__} | cuda {torch.version.cuda}")

## 4. Data download
This cell fetches the official Yelp Polarity archive (Zhang, Zhao and LeCun 2015, about 160 MB) if it's missing, and records the
SHA-256 of `train.csv` and `test.csv`.

**If the download fails:** get `yelp_review_polarity_csv.tgz` another way (a browser, or the Kaggle copy). Put `train.csv`,
`test.csv` and `readme.txt` in `task2_sentiment/data/raw/yelp_review_polarity_csv/`, then rerun this cell.

In [ ]:
def download_yelp() -> None:
    RAW.parent.mkdir(parents=True, exist_ok=True)
    tgz = RAW.parent / "yelp_review_polarity_csv.tgz"
    if not tgz.exists():
        part = tgz.with_name(tgz.name + ".part")
        print(f"downloading {cfg.data_url} ...")
        with urllib.request.urlopen(cfg.data_url) as resp, open(part, "wb") as out:
            shutil.copyfileobj(resp, out, length=1 << 20)
        part.rename(tgz)
    with tarfile.open(tgz) as tar:
        names = [m for m in tar.getmembers() if m.isfile()
                 and Path(m.name).name in ("train.csv", "test.csv", "readme.txt")]
        for m in names:                                   # extract only the three known files
            m.name = Path(m.name).name
            tar.extract(m, RAW)


if not ((RAW / "train.csv").is_file() and (RAW / "test.csv").is_file()):
    download_yelp()
DATA_SHA = {}
for name in ("train.csv", "test.csv"):
    side = RAW / f"{name}.sha256"
    if not side.exists():
        side.write_text(f"{sha256_file(RAW / name)}  {name}\n", encoding="utf-8")
    DATA_SHA[name] = side.read_text(encoding="utf-8").split()[0]
    print(f"{name}: {(RAW / name).stat().st_size:,} bytes | sha256 {DATA_SHA[name]}")
if (RAW / "readme.txt").exists():
    readme = (RAW / "readme.txt").read_text(encoding="utf-8", errors="replace")
    print("\nreadme (labels):", " ".join(l.strip() for l in readme.splitlines()
                                         if re.search(r"polarity|negative|positive", l, re.I))[:600])

## 5. The shared split (team-owned, SHARED with Ayush)
- **The split:** 100,000 training and 20,000 validation rows, stratified and drawn from the official training file with seed 42.
  The test set is the full official 38,000 rows.
- **The files are created once,** in `task2_sentiment/data/splits/`, then **loaded, never redrawn**.
  Commit them and give Ayush the same files. Accuracy comparisons and the McNemar test are only valid if you both score the same test rows.

In [ ]:
def read_yelp(path: Path) -> pd.DataFrame:
    """The CSVs have no header: "label","text". Doubled quotes are CSV escaping (pandas undoes
    them); newlines are stored as the two characters backslash + n. keep_default_na=False
    keeps a review that literally says "NA" as text instead of a missing value."""
    df = pd.read_csv(path, header=None, names=["label", "text"], dtype={"text": str},
                     keep_default_na=False, encoding="utf-8")
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    return df


def make_split(labels: np.ndarray, n_train: int, n_val: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Stratified, disjoint train / validation row indices into the official training file,
    drawn only from rows with a valid label (1 or 2)."""
    idx = np.flatnonzero(np.isin(labels, [1, 2]))
    tr, va = train_test_split(idx, train_size=n_train, test_size=n_val, stratify=labels[idx], random_state=seed)
    return np.sort(tr), np.sort(va)


train_df, test_df = read_yelp(RAW / "train.csv"), read_yelp(RAW / "test.csv")
paths = {k: SPLITS / f"{k}_idx.npy" for k in ("train", "val", "test")}
if all(p.exists() for p in paths.values()):
    split_idx = {k: np.load(p) for k, p in paths.items()}
    print("loaded the existing split files (not redrawn)")
else:
    SPLITS.mkdir(parents=True, exist_ok=True)
    labels = train_df["label"].fillna(-1).to_numpy()
    tr, va = make_split(labels, CONFIG["n_train"], CONFIG["n_val"], CONFIG["split_seed"])
    split_idx = {"train": tr, "val": va, "test": np.arange(len(test_df))}
    for k, p in paths.items():
        np.save(p, split_idx[k].astype(np.int64))
    info = {"method": "sklearn train_test_split, stratified on the label, then sorted",
            "seed": CONFIG["split_seed"], "source": "official yelp_review_polarity_csv",
            "sizes": {k: int(len(v)) for k, v in split_idx.items()},
            "train_rows_without_valid_label_excluded": int((~np.isin(labels, [1, 2])).sum()),
            "train_csv_sha256": DATA_SHA["train.csv"], "test_csv_sha256": DATA_SHA["test.csv"],
            "files": {p.name: sha256_file(p) for p in paths.values()},
            "made_by": "task2_sentiment/sneha/src/task2_sentiment_sneha.ipynb, section 5"}
    (SPLITS / "split_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("CREATED the split files. Commit task2_sentiment/data/splits/ and share it with Ayush.")

assert len(np.intersect1d(split_idx["train"], split_idx["val"])) == 0, "train and val overlap"
assert len(split_idx["test"]) == len(test_df), "the test split must be the full official test set"
SPLIT_SHA = {p.name: sha256_file(p) for p in paths.values()}
if SMOKE:                                               # smoke only: the first rows of each split
    split_idx = {k: v[: cfg.smoke_rows[k]] for k, v in split_idx.items()}
raw = {"train": train_df.iloc[split_idx["train"]].reset_index(drop=True),
       "val": train_df.iloc[split_idx["val"]].reset_index(drop=True),
       "test": test_df.iloc[split_idx["test"]].reset_index(drop=True)}
del train_df
print({k: len(v) for k, v in raw.items()}, "| split files sha256:", {k: v[:12] for k, v in SPLIT_SHA.items()})

## 6. Preprocessing (brief 2.1)
- **Two separate pipelines,** so they never contaminate each other:
  - **Word pipeline** (meanpool and BiGRU): unescape the stored `\n` → lowercase → drop apostrophes (so *don't* → *dont*, keeping the negation
    as one token) → strip punctuation and special characters → whitespace tokens → remove stopwords → Snowball stemming →
    a vocabulary built from the training split only.
  - **Character pipeline** (CharCNN): its own fixed alphabet. It never touches the word vocabulary.
- **Stopwords:** scikit-learn's English list, minus `keep_words`. Standard lists contain *not*, *no*, *never* and *but*, which carry
  sentiment ("not good"), so those are kept.
- **Malformed entries** are dropped from **training and validation only**: a missing or invalid label, text that's empty after cleaning, a duplicate
  within training, or a training/validation review that also appears in the test set (leakage). Test rows are never dropped, because McNemar
  and the team comparison need identical rows. A test review that's empty after cleaning becomes a single `<unk>`.

In [ ]:
PAD, UNK = 0, 1
STOPWORDS = frozenset(ENGLISH_STOP_WORDS) - set(cfg.keep_words)
_stemmer = SnowballStemmer("english")
_stem = lru_cache(maxsize=None)(_stemmer.stem)
_NON_ALNUM = re.compile(r"[^a-z0-9\s]")


def unescape(raw: str) -> str:
    """The dataset stores newlines as the two characters backslash + n."""
    return raw.replace("\\n", "\n").replace('\\"', '"')


def clean_words(raw: str, remove_stopwords: bool, stem: bool) -> list[str]:
    t = unescape(raw).lower().replace("'", "").replace("’", "")
    tokens = _NON_ALNUM.sub(" ", t).split()
    if remove_stopwords:
        tokens = [w for w in tokens if w not in STOPWORDS]
    if stem:
        tokens = [_stem(w) for w in tokens]
    return tokens


def build_vocab(token_lists, max_vocab: int) -> list[str]:
    """Most frequent types first, ties broken alphabetically, so the vocabulary is deterministic.
    Index 0 is <pad>, 1 is <unk>."""
    counts = Counter(w for toks in token_lists for w in toks)
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:max_vocab]
    return ["<pad>", "<unk>"] + [w for w, _ in ranked]


def encode_words(token_lists, vocab: list[str], max_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    """(N, max_len) ids padded with 0, and true lengths. An empty review becomes one <unk>,
    so every sequence has length >= 1 (packed GRU sequences cannot be empty)."""
    index = {w: i for i, w in enumerate(vocab)}
    ids = np.zeros((len(token_lists), max_len), dtype=np.int64)
    lengths = np.zeros(len(token_lists), dtype=np.int64)
    for r, toks in enumerate(token_lists):
        seq = [index.get(w, UNK) for w in toks[:max_len]] or [UNK]
        ids[r, :len(seq)] = seq
        lengths[r] = len(seq)
    return torch.from_numpy(ids), torch.from_numpy(lengths)


# Character pipeline (Zhang et al. 2015). The paper's list has '-' twice: dedupe, keep the order.
_RAW_ALPHABET = "abcdefghijklmnopqrstuvwxyz0123456789-,;.!?:'\"/\\|_@#$%^&*~`+-=<>()[]{}\n"
ALPHABET = list(dict.fromkeys(_RAW_ALPHABET))
N_CHARS = len(ALPHABET)                       # 69; index N_CHARS = padding / out-of-alphabet
assert N_CHARS == 69
_CHAR_LUT = np.full(0x110000, N_CHARS, dtype=np.int8)
for i, ch in enumerate(ALPHABET):
    _CHAR_LUT[ord(ch)] = i


def encode_chars(raw_texts, length: int) -> torch.Tensor:
    """(N, length) int8 character indices: unescape, lowercase, truncate / pad with N_CHARS."""
    out = np.full((len(raw_texts), length), N_CHARS, dtype=np.int8)
    for r, raw in enumerate(raw_texts):
        s = unescape(raw).lower()[:length]
        if s:
            cps = np.frombuffer(s.encode("utf-32-le"), dtype=np.uint32)
            out[r, :len(cps)] = _CHAR_LUT[cps]
    return torch.from_numpy(out)


def slice_masks(raw_texts) -> dict[str, np.ndarray]:
    """The four robustness slices, on the raw review text (stored newlines unescaped)."""
    texts = [unescape(t) for t in raw_texts]
    n_tok = np.array([len(t.split()) for t in texts])
    neg = re.compile(r"\b(not|no|never|cannot)\b|\w+n't\b", re.I)
    con = re.compile(r"\b(but|however|although|though)\b", re.I)
    return {"short": n_tok < cfg.short_max_tokens, "long": n_tok > cfg.long_min_tokens,
            "negation": np.array([bool(neg.search(t)) for t in texts]),
            "contrast": np.array([bool(con.search(t)) for t in texts])}

## 7. Models (brief 2.2)
- **Every embedding is `nn.Embedding`, randomly initialised and trained.** Nothing is pretrained.
- **Each model outputs one logit,** and p(positive) = sigmoid(logit).

In [ ]:
class MeanPool(nn.Module):
    """M1 baseline: a deep averaging network. Masked mean of word embeddings over the real
    (non-pad) positions, then two ReLU layers. No word order at all."""

    def __init__(self, vocab_size, emb_dim=128, hidden=256, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.mlp = nn.Sequential(nn.Linear(emb_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, ids, lengths):
        mask = (ids != PAD).unsqueeze(-1).float()
        mean = (self.emb(ids) * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return self.mlp(mean).squeeze(-1)


class BiGRUMax(nn.Module):
    """M2: one bidirectional GRU layer over packed sequences (padding never enters the
    recurrence), then max-pooling over time, ignoring padded positions."""

    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * hidden, 1)

    def forward(self, ids, lengths):
        T = ids.size(1)
        packed = pack_padded_sequence(self.emb(ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=T)
        valid = torch.arange(T, device=ids.device)[None, :] < lengths.to(ids.device)[:, None]
        h = h.masked_fill(~valid.unsqueeze(-1), float("-inf"))   # padding cannot win the max
        return self.out(self.drop(h.max(dim=1).values)).squeeze(-1)


class CharCNN(nn.Module):
    """M3: the small character CNN of Zhang, Zhao and LeCun (2015), on 1014 characters.
    Length trace: 1014 -7+1-> 1008 /3-> 336 -7+1-> 330 /3-> 110 -2-> 108 -2-> 106 -2-> 104
    -2-> 102 /3-> 34, and 256 x 34 = 8704 flattened features."""

    FLATTEN = 256 * 34

    def __init__(self, input_mode="onehot", emb_dim=16, filters=256, fc=512, dropout=0.5, length=1014):
        super().__init__()
        if (filters, length) != (256, 1014):
            raise ValueError("the flatten size 8704 assumes 256 filters and 1014 characters")
        self.input_mode = input_mode
        if input_mode == "embedding":     # a learned character embedding; padding row stays zero
            self.emb = nn.Embedding(N_CHARS + 1, emb_dim, padding_idx=N_CHARS)
            in_ch = emb_dim
        elif input_mode == "onehot":      # the paper: one-hot; padding / unknown = all-zero column
            in_ch = N_CHARS
        else:
            raise ValueError(f"charcnn_input must be onehot or embedding, not {input_mode!r}")
        k = [7, 7, 3, 3, 3, 3]
        pool = [True, True, False, False, False, True]
        layers, c = [], in_ch
        for ki, pi in zip(k, pool):
            layers += [nn.Conv1d(c, filters, ki), nn.ReLU()] + ([nn.MaxPool1d(3)] if pi else [])
            c = filters
        self.conv = nn.Sequential(*layers)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(self.FLATTEN, fc), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(fc, fc), nn.ReLU(), nn.Dropout(dropout), nn.Linear(fc, 1))

    def forward(self, chars):
        x = chars.long()
        if self.input_mode == "embedding":
            x = self.emb(x).transpose(1, 2)                                   # (B, emb_dim, L)
        else:
            x = F.one_hot(x, N_CHARS + 1)[..., :N_CHARS].transpose(1, 2).float()  # (B, 69, L)
        return self.fc(self.conv(x)).squeeze(-1)


def build_model(name: str, vocab_size: int) -> nn.Module:
    if name in ("meanpool", "meanpool_ablation"):
        return MeanPool(vocab_size, cfg.emb_dim, cfg.mlp_hidden, cfg.dropout)
    if name == "bigru":
        return BiGRUMax(vocab_size, cfg.emb_dim, cfg.gru_hidden, cfg.dropout)
    if name == "charcnn":
        return CharCNN(cfg.charcnn_input, cfg.charcnn_emb_dim, cfg.charcnn_filters, cfg.charcnn_fc,
                       cfg.charcnn_dropout, cfg.char_len)
    raise ValueError(name)


def num_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

## 8. Metrics (brief, Task 2 list; definitions SHARED with Ayush)
- **Classification:** accuracy; precision, recall and F1 (macro, micro and weighted); the confusion matrix; ROC-AUC and PR-AUC on p(positive); MCC; and the Brier score.
- **ECE:** 15 equal-width bins over [0, 1] of confidence = max(p, 1 − p). A prediction counts as correct when (p ≥ 0.5) equals the label.
  ECE is the bin-weighted mean of |accuracy − confidence|.
- **Bootstrap:** 1000 resamples of the test set with replacement, seed 42, giving 95% percentile intervals for accuracy, macro-F1 and MCC.
- **McNemar:** baseline vs each experimental model, on identical test rows, with continuity correction (statsmodels, `exact=False`).
- **Slices:** macro-F1 and error rate on each of the four slices (they overlap), with the size of each.

In [ ]:
def full_report(y, p) -> dict:
    pred = (p >= 0.5).astype(int)
    r = {"accuracy": accuracy_score(y, pred)}
    for avg in ("macro", "micro", "weighted"):
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average=avg, zero_division=0)
        r.update({f"precision_{avg}": pr, f"recall_{avg}": rc, f"f1_{avg}": f1})
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    r.update({"cm_tn": int(tn), "cm_fp": int(fp), "cm_fn": int(fn), "cm_tp": int(tp)})
    both = len(np.unique(y)) == 2
    r["roc_auc"] = roc_auc_score(y, p) if both else float("nan")
    r["pr_auc"] = average_precision_score(y, p) if both else float("nan")
    r["mcc"] = matthews_corrcoef(y, pred)
    r["brier"] = brier_score_loss(y, p)
    r["ece"] = expected_calibration_error(y, p, cfg.ece_bins)
    return r


def expected_calibration_error(y, p, n_bins: int = 15) -> float:
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    b = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    ece = 0.0
    for k in range(n_bins):
        m = b == k
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def reliability_bins(y, p, n_bins: int = 15):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    b = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    rows = [(k, conf[b == k].mean(), correct[b == k].mean(), (b == k).sum()) for k in range(n_bins) if (b == k).any()]
    return pd.DataFrame(rows, columns=["bin", "confidence", "accuracy", "n"])


def _binary_scores(tp, fp, fn, tn):
    """Accuracy, macro-F1 and MCC from (vectorised) confusion counts; 0 where undefined."""
    with np.errstate(divide="ignore", invalid="ignore"):
        n = tp + fp + fn + tn
        f1_pos = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
        f1_neg = np.where(2 * tn + fn + fp > 0, 2 * tn / (2 * tn + fn + fp), 0.0)
        denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = np.where(denom > 0, (tp * tn - fp * fn) / denom, 0.0)
    return (tp + tn) / n, (f1_pos + f1_neg) / 2, mcc


def bootstrap_ci(y, pred, n_boot: int = 1000, seed: int = 42, chunk: int = 50) -> dict:
    rng = np.random.default_rng(seed)
    y, pred = np.asarray(y), np.asarray(pred)
    acc, f1, mcc = [], [], []
    for start in range(0, n_boot, chunk):
        idx = rng.integers(0, len(y), size=(min(chunk, n_boot - start), len(y)))
        yt, yp = y[idx], pred[idx]
        tp = ((yt == 1) & (yp == 1)).sum(1).astype(float); tn = ((yt == 0) & (yp == 0)).sum(1).astype(float)
        fp = ((yt == 0) & (yp == 1)).sum(1).astype(float); fn = ((yt == 1) & (yp == 0)).sum(1).astype(float)
        a, f, m = _binary_scores(tp, fp, fn, tn)
        acc.append(a); f1.append(f); mcc.append(m)
    out = {}
    for name, v in (("acc", acc), ("f1_macro", f1), ("mcc", mcc)):
        lo, hi = np.percentile(np.concatenate(v), [2.5, 97.5])
        out[f"{name}_ci_lo"], out[f"{name}_ci_hi"] = float(lo), float(hi)
    return out


def mcnemar_test(y, pred_a, pred_b) -> tuple[float, float]:
    a, b = pred_a == y, pred_b == y
    table = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
    res = mcnemar(table, exact=False, correction=True)
    return float(res.statistic), float(res.pvalue)


def slice_report(y, pred, masks) -> dict:
    out = {}
    for name, m in masks.items():
        n = int(m.sum())
        out[f"slice_{name}_n"] = n
        out[f"slice_{name}_f1"] = f1_score(y[m], pred[m], average="macro", zero_division=0) if n else float("nan")
        out[f"slice_{name}_err"] = float((y[m] != pred[m]).mean()) if n else float("nan")
    return out


def select_errors(y, p, masks, worst_slice: str, band=(0.45, 0.55), k: int = 5) -> tuple[list, int]:
    """The 20-error review: 5 confident FPs, 5 confident FNs, 5 near-threshold errors (fallback:
    closest to 0.5), 5 from the worst slice. Returns ([(category, row), ...], #near errors in band)."""
    pred = (p >= 0.5).astype(int)
    err = np.flatnonzero(pred != y)
    chosen, used = [], set()

    def take(rows, cat):
        for r in rows:
            if len([c for c in chosen if c[0] == cat]) == k:
                break
            if r not in used:
                chosen.append((cat, int(r))); used.add(r)

    fp = err[(pred[err] == 1) & (y[err] == 0)]
    take(fp[np.argsort(-p[fp], kind="stable")], "confident false positive")
    fn = err[(pred[err] == 0) & (y[err] == 1)]
    take(fn[np.argsort(p[fn], kind="stable")], "confident false negative")
    by_dist = err[np.argsort(np.abs(p[err] - 0.5), kind="stable")]
    in_band = int(((p[err] >= band[0]) & (p[err] <= band[1])).sum())
    take(by_dist, "near threshold")
    sl = err[masks[worst_slice][err]]
    take(sl[np.argsort(-np.abs(p[sl] - 0.5), kind="stable")], f"slice: {worst_slice}")
    return chosen, in_band

## 9. Unit tests (must pass before training)
These run on the CPU with small synthetic inputs, and stop the notebook if any test fails.

In [ ]:
def run_tests() -> None:
    torch_state = torch.get_rng_state()

    def t_alphabet_and_charcnn_flatten():
        assert N_CHARS == 69 and ALPHABET.count("-") == 1 and "\n" in ALPHABET
        for mode in ("onehot", "embedding"):
            m = CharCNN(mode).eval()
            x = torch.full((2, 1014), N_CHARS, dtype=torch.int8)
            assert m.conv(F.one_hot(x.long(), N_CHARS + 1)[..., :N_CHARS].transpose(1, 2).float()
                          if mode == "onehot" else m.emb(x.long()).transpose(1, 2)).flatten(1).shape[1] == 8704
            assert m(x).shape == (2,)

    def t_char_encoding():
        e = encode_chars(['Hi!\\nZé', ""], 10)
        assert e.dtype == torch.int8 and e.shape == (2, 10)
        expect = [ALPHABET.index(c) for c in "hi!\nz"] + [N_CHARS] * 5    # é is outside the alphabet
        assert e[0].tolist() == expect and e[1].tolist() == [N_CHARS] * 10
        assert encode_chars(["a" * 2000], 1014).shape == (1, 1014)

    def t_word_cleaning():
        raw = "The food wasn't good, NOT great!\\nI don't like it... but very cheap."
        toks = clean_words(raw, remove_stopwords=True, stem=True)
        for w in ("not", "dont", "but", "veri"):
            assert w in toks, (w, toks)
        assert "the" not in toks and "!" not in "".join(toks) and "\\n" not in "".join(toks)
        plain = clean_words(raw, remove_stopwords=False, stem=False)
        assert "the" in plain and "wasnt" in plain and "food" in plain

    def t_vocab_and_encoding():
        v = build_vocab([["b", "a", "a"], ["c", "b", "a"]], max_vocab=2)
        assert v == ["<pad>", "<unk>", "a", "b"]                            # capped, by frequency
        ids, lens = encode_words([["a", "zzz", "b"], [], ["a"] * 10], v, max_len=4)
        assert ids[0].tolist() == [2, 1, 3, 0] and lens.tolist() == [3, 1, 4]
        assert ids[1].tolist() == [UNK, 0, 0, 0]                            # empty -> one <unk>

    def t_padding_does_not_change_outputs():
        torch.manual_seed(0)
        for m in (MeanPool(50), BiGRUMax(50)):
            m.eval()
            a = torch.tensor([[5, 6, 7, 0]]); b = torch.tensor([[5, 6, 7, 0, 0, 0, 0, 0]])
            la, lb = m(a, torch.tensor([3])), m(b, torch.tensor([3]))
            assert torch.allclose(la, lb, atol=1e-6), type(m).__name__

    def t_param_counts():
        V, d, h, g = 1000, 128, 256, 128
        assert num_params(MeanPool(V, d, h)) == V * d + (d * h + h) + (h * h + h) + (h + 1)
        assert num_params(BiGRUMax(V, d, g)) == V * d + 2 * 3 * (d * g + g * g + 2 * g) + (2 * g + 1)
        conv = 69 * 256 * 7 + 256 + 256 * 256 * 7 + 256 + 4 * (256 * 256 * 3 + 256)
        assert num_params(CharCNN()) == conv + (8704 * 512 + 512) + (512 * 512 + 512) + 513

    def t_metrics():
        y = np.array([0, 0, 1, 1]); p = np.array([0.1, 0.6, 0.8, 0.4])
        r = full_report(y, p)
        assert r["accuracy"] == 0.5 and (r["cm_tn"], r["cm_fp"], r["cm_fn"], r["cm_tp"]) == (1, 1, 1, 1)
        # ECE by hand: confidences .9,.6,.8,.6 -> bins 13, 9, 12, 9; correct 1,0,1,0
        assert abs(expected_calibration_error(y, p, 15) - (0.25 * 0.1 + 0.5 * 0.6 + 0.25 * 0.2)) < 1e-12
        rng = np.random.default_rng(1)
        yy = rng.integers(0, 2, 500); pp = np.where(rng.random(500) < 0.8, yy, 1 - yy)
        tp, tn = float(((yy == 1) & (pp == 1)).sum()), float(((yy == 0) & (pp == 0)).sum())
        fp, fn = float(((yy == 0) & (pp == 1)).sum()), float(((yy == 1) & (pp == 0)).sum())
        a, f, m = _binary_scores(tp, fp, fn, tn)
        assert abs(a - accuracy_score(yy, pp)) < 1e-12 and abs(f - f1_score(yy, pp, average="macro")) < 1e-12
        assert abs(m - matthews_corrcoef(yy, pp)) < 1e-12
        ci = bootstrap_ci(yy, pp, n_boot=200, seed=42)
        assert ci["acc_ci_lo"] < accuracy_score(yy, pp) < ci["acc_ci_hi"]
        assert bootstrap_ci(yy, pp, 200, 42) == ci                          # seeded
        # McNemar: a right / b wrong on 10 rows, the reverse on 2 -> (|10-2|-1)^2 / 12
        yt = np.ones(20, int); pa = np.r_[np.ones(10), np.zeros(2), np.ones(8)].astype(int)
        pb = np.r_[np.zeros(10), np.ones(2), np.ones(8)].astype(int)
        assert abs(mcnemar_test(yt, pa, pb)[0] - 49 / 12) < 1e-9

    def t_slices():
        m = slice_masks(["It is not good", "I didn't like it", "nobody pressed the button",
                         "good but pricey", "Although fine", "word " * 250])
        assert m["negation"].tolist() == [True, True, False, False, False, False]
        assert m["contrast"].tolist() == [False, False, False, True, True, False]
        assert m["short"].tolist()[:5] == [True] * 5 and m["long"].tolist() == [False] * 5 + [True]

    def t_select_errors():
        y = np.array([0] * 10 + [1] * 10)
        p = np.array([.99, .98, .97, .96, .95, .94, .51, .52, .1, .2] + [.01, .02, .03, .04, .05, .06, .49, .9, .8, .7])
        masks = {"short": np.ones(20, bool)}
        chosen, in_band = select_errors(y, p, masks, "short")
        cats = Counter(c for c, _ in chosen)
        assert cats["confident false positive"] == 5 and [r for c, r in chosen][:5] == [0, 1, 2, 3, 4]
        assert cats["confident false negative"] == 5 and [r for c, r in chosen][5:10] == [10, 11, 12, 13, 14]
        near = [r for c, r in chosen if c == "near threshold"]
        assert in_band == 3 and near[:3] == [6, 16, 7]                      # 3 inside [0.45, 0.55], closest first
        assert len(near) == 5                                                # filled with the next closest to 0.5
        assert len({r for _, r in chosen}) == len(chosen)                    # no row picked twice

    def t_split():
        labels = np.array([1] * 600 + [2] * 400)
        tr, va = make_split(labels, 100, 50, 42)
        assert len(tr) == 100 and len(va) == 50 and not np.intersect1d(tr, va).size
        assert (labels[tr] == 2).sum() == 40 and (labels[va] == 2).sum() == 20   # stratified
        assert np.array_equal(make_split(labels, 100, 50, 42)[0], tr)            # seeded

    def t_gpu_checks():
        archs = ["sm_50", "sm_60", "sm_70", "sm_75", "sm_80", "sm_86", "sm_90"]  # torch 2.4.1
        assert arch_supported((8, 9), archs) and not arch_supported((12, 0), archs)
        original = torch.cuda.is_available
        torch.cuda.is_available = lambda: False
        try:
            assert get_device("auto").type == "cpu"
            try:
                get_device("cuda")
                raise AssertionError("cuda must never fall back to the CPU silently")
            except RuntimeError:
                pass
        finally:
            torch.cuda.is_available = original

    tests = [t_alphabet_and_charcnn_flatten, t_char_encoding, t_word_cleaning, t_vocab_and_encoding,
             t_padding_does_not_change_outputs, t_param_counts, t_metrics, t_slices, t_select_errors,
             t_split, t_gpu_checks]
    t0 = time.perf_counter()
    for t in tests:
        t()
        print(f"[ok] {t.__name__[2:]}")
    torch.set_rng_state(torch_state)
    print(f"all {len(tests)} test groups passed in {time.perf_counter() - t0:.1f}s")


run_tests()

## 10. Preprocessing run and data analysis (brief 2.1.1–2.1.4)
- **This cell produces:**
  - the cleaning log
  - the three required plots: review length distribution, class distribution and class balance
  - the vocabulary, written to `data_processed/`
- **For the ablation,** a second word pipeline runs with stopword removal and stemming off.

In [ ]:
def preprocess() -> dict:
    logger, log_path = open_run_log("preprocess")
    try:
        snapshot_config(log_path)
        logger.info(f"train.csv sha256 {DATA_SHA['train.csv']} | test.csv sha256 {DATA_SHA['test.csv']} | "
                    f"split files {json.dumps(SPLIT_SHA)}")
        test_texts = set(raw["test"]["text"])
        kept, report = {}, {}
        for name in ("train", "val", "test"):
            df = raw[name].copy()
            df["y"] = df["label"] - 1
            n0 = len(df)
            if name == "test":
                bad = ~df["y"].isin([0, 1])
                assert not bad.any(), f"{int(bad.sum())} test rows have no valid label"
                report[name] = {"rows": n0}
            else:
                drops = {}
                m = ~df["y"].isin([0, 1]); drops["invalid_or_missing_label"] = int(m.sum()); df = df[~m]
                m = df["text"].map(lambda t: not clean_words(t, False, False)); drops["empty_after_cleaning"] = int(m.sum()); df = df[~m]
                m = df["text"].duplicated(); drops["duplicate_text"] = int(m.sum()); df = df[~m]
                m = df["text"].isin(test_texts); drops["also_in_test_set"] = int(m.sum()); df = df[~m]
                report[name] = {"rows_before": n0, **drops, "rows_after": len(df)}
            kept[name] = df.reset_index(drop=True)
            logger.info(f"{name}: {json.dumps(report[name])}")
        dup = kept["val"]["text"].isin(set(kept["train"]["text"]))   # same review on both sides of the split
        report["val"]["also_in_train"] = int(dup.sum())
        report["val"]["rows_after"] = int((~dup).sum())
        kept["val"] = kept["val"][~dup].reset_index(drop=True)
        logger.info(f"val: dropped {int(dup.sum())} reviews that also appear in train")

        # analysis plots (brief 2.1.1)
        lens = {n: kept[n]["text"].map(lambda t: len(unescape(t).split())) for n in kept}
        fig, ax = plt.subplots(figsize=(7, 4))
        for lab, nm in ((0, "negative"), (1, "positive")):
            ax.hist(lens["train"][kept["train"]["y"] == lab].clip(upper=1000), bins=60, alpha=0.6, label=nm)
        ax.set_xlabel("review length (whitespace tokens, clipped at 1000)"); ax.set_ylabel("reviews")
        ax.set_title("Review length distribution (training split)"); ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(OUTPUTS / "length_distribution.png", dpi=150); plt.close(fig)
        counts = pd.DataFrame({n: kept[n]["y"].value_counts().reindex([0, 1], fill_value=0) for n in kept},
                              ).rename(index={0: "negative", 1: "positive"})
        ax = counts.T.plot.bar(figsize=(6, 4), rot=0); ax.set_ylabel("reviews"); ax.set_title("Class distribution per split")
        ax.grid(alpha=0.3, axis="y"); plt.tight_layout(); plt.savefig(OUTPUTS / "class_distribution.png", dpi=150); plt.close()
        share = counts / counts.sum()
        ax = share.T.plot.bar(stacked=True, figsize=(6, 4), rot=0); ax.axhline(0.5, ls="--", c="k", lw=0.8)
        ax.set_ylabel("share of reviews"); ax.set_title("Class balance per split")
        plt.tight_layout(); plt.savefig(OUTPUTS / "class_balance.png", dpi=150); plt.close()
        length_stats = {n: {k: float(v) for k, v in lens[n].describe(percentiles=[.5, .9, .99]).items()} for n in lens}
        logger.info(f"length stats {json.dumps(length_stats)}")
        logger.info(f"class counts {counts.to_dict()}")

        # word pipelines: main (stopwords + stemming as configured) and the ablation (both off)
        pipes = {"main": (cfg.remove_stopwords, cfg.stem)}
        if cfg.ablation:
            pipes["ablation"] = (False, False)
        data = {"y": {n: torch.tensor(kept[n]["y"].to_numpy(), dtype=torch.float32) for n in kept},
                "texts": {n: kept[n]["text"].tolist() for n in kept}, "words": {}, "vocab": {}}
        for pipe, (sw, st) in pipes.items():
            t0 = time.perf_counter()
            toks = {n: [clean_words(t, sw, st) for t in data["texts"][n]] for n in kept}
            vocab = build_vocab(toks["train"], cfg.max_vocab)
            data["vocab"][pipe] = vocab
            data["words"][pipe] = {n: encode_words(toks[n], vocab, cfg.max_len) for n in kept}
            index = set(vocab)
            oov = {n: float(np.mean([w not in index for t in toks[n] for w in t] or [0])) for n in kept}
            trunc = {n: float(np.mean([len(t) > cfg.max_len for t in toks[n]])) for n in kept}
            (PROCESSED / f"vocab_{pipe}.json").write_text(json.dumps(vocab, ensure_ascii=False), encoding="utf-8")
            logger.info(f"word pipeline '{pipe}' (stopwords removed {sw}, stemming {st}): vocab {len(vocab)} ids | "
                        f"OOV rate {json.dumps(oov)} | truncated at {cfg.max_len}: {json.dumps(trunc)} | "
                        f"{time.perf_counter() - t0:.1f}s")
            example = [{"raw": data["texts"]["train"][i][:300], "tokens": toks["train"][i][:40]} for i in range(5)]
            (PROCESSED / f"examples_{pipe}.json").write_text(json.dumps(example, indent=1, ensure_ascii=False), encoding="utf-8")
        t0 = time.perf_counter()
        data["chars"] = {n: encode_chars(data["texts"][n], cfg.char_len) for n in kept}
        unknown = float((data["chars"]["train"] == N_CHARS).float().mean())
        logger.info(f"char pipeline: {N_CHARS} characters, length {cfg.char_len}, share of padding/unknown "
                    f"positions in train {unknown:.3f} | {time.perf_counter() - t0:.1f}s")
        data["masks"] = slice_masks(data["texts"]["test"])
        logger.info("test slices " + json.dumps({k: int(v.sum()) for k, v in data["masks"].items()}))
        data["report"] = report
        (PROCESSED / "preprocessing_summary.json").write_text(json.dumps(
            {"cleaning": report, "length_stats": length_stats, "class_counts": counts.to_dict(),
             "stopwords_removed": sorted(STOPWORDS) if cfg.remove_stopwords else [], "kept_words": cfg.keep_words,
             "alphabet": "".join(ALPHABET)}, indent=1, ensure_ascii=False), encoding="utf-8")
        data["log"] = rel(log_path)
        return data
    except BaseException:
        logger.exception("preprocessing stopped with an error")
        raise
    finally:
        close_run_log(logger)


data = preprocess()
display(pd.DataFrame(data["report"]).T.fillna("").astype(str))
for f in ("length_distribution.png", "class_distribution.png", "class_balance.png"):
    display(Image(OUTPUTS / f))

## 11. Pre-flight
- **Checks:** versions, the GPU and free disk.
- **Time estimate:** it times 20 training steps of each model and estimates the full run from them.
- **For the real run:** it must say `PREFLIGHT PASSED` on the RTX 4090.

In [ ]:
def model_inputs(name: str, split: str, pipe: str = "main"):
    if name == "charcnn":
        return (data["chars"][split],)
    return data["words"][pipe][split]


def preflight(n_steps: int = 20) -> None:
    failed = []

    def report(ok, what, hard=True):
        print(f"[{'ok  ' if ok else ('FAIL' if hard else 'warn')}] {what}")
        if not ok and hard:
            failed.append(what)

    report((3, 10) <= sys.version_info[:2] <= (3, 12), f"python {platform.python_version()}", hard=False)
    report(torch.__version__.split("+")[0] == "2.4.1", f"torch {torch.__version__}, cuda build {torch.version.cuda}", hard=False)
    report(True, f"device {DEVICE.type}: {device_name(DEVICE)} | cpu {cpu_name()}")
    report(shutil.disk_usage(ROOT).free / 1e9 > 3, f"free disk {shutil.disk_usage(ROOT).free / 1e9:.1f} GB")
    n_train = len(data["y"]["train"])
    steps_per_epoch = math.ceil(n_train / cfg.batch_size)
    total = 0.0
    for name in cfg.models + (["meanpool_ablation"] if cfg.ablation else []):
        pipe = "ablation" if name == "meanpool_ablation" else "main"
        m = build_model(name, len(data["vocab"][pipe])).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=cfg.lr)
        xs = [x[: cfg.batch_size].to(DEVICE) for x in model_inputs(name, "train", pipe)]
        y = data["y"]["train"][: cfg.batch_size].to(DEVICE)
        times = []
        for i in range(n_steps + 3):
            sync(DEVICE); t0 = time.perf_counter()
            loss = F.binary_cross_entropy_with_logits(m(*xs), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); loss.item(); sync(DEVICE)
            if i >= 3:
                times.append(time.perf_counter() - t0)
        step_s = float(np.median(times))
        est = step_s * steps_per_epoch * cfg.max_epochs * 1.2
        total += est
        report(math.isfinite(loss.item()), f"{name:18s} {num_params(m):>10,} params | {step_s * 1000:6.1f} ms/step | "
               f"<= {est / 60:.1f} min for {cfg.max_epochs} epochs")
        del m, opt
    print(f"       estimated total training time: <= {total / 60:.0f} min ({steps_per_epoch} steps/epoch)")
    if DEVICE.type == "cpu" and total > 3600:
        report(False, "this would take hours on the CPU; use a lab RTX 4090", hard=False)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("\nPREFLIGHT " + (f"FAILED ({len(failed)} problem(s) above)" if failed else "PASSED"))
    assert not failed, "pre-flight failed: fix the FAIL lines before training"


preflight()

## 12. Train the three models, plus the ablation (brief 2.2.1)
- **Training setup:** Adam (lr 1e-3), batch 64, up to 5 epochs, binary cross-entropy on one logit.
- **Early stopping:** on validation macro-F1, with patience 2. The best epoch is checkpointed.
- **Logging:** every step goes to the model's raw log.
- **Recorded per model:** device, CPU, training time, examples/sec and peak memory, all measured at runtime.

In [ ]:
def predict(model, inputs, bs: int = 512) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(inputs[0]), bs):
            out.append(torch.sigmoid(model(*[x[i:i + bs].to(DEVICE) for x in inputs])).float().cpu())
    return torch.cat(out).numpy()


def train_model(name: str) -> dict:
    pipe = "ablation" if name == "meanpool_ablation" else "main"
    ckpt, summary_path = CKPT_DIR / f"{name}_best.pt", OUTPUTS / f"{name}_train_summary.json"
    if RESUME and summary_path.exists() and ckpt.exists():
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        if s["config_sha256"] == CFG_HASH:
            print(f"{name}: already trained with these settings, reusing {rel(ckpt)}")
            return s
    set_seeds(cfg.seed)
    logger, log_path = open_run_log(f"{name}_train")
    try:
        snapshot_config(log_path)
        vocab = data["vocab"][pipe]
        model = build_model(name, len(vocab)).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
        n_params = num_params(model)
        train_x, val_x = model_inputs(name, "train", pipe), model_inputs(name, "val", pipe)
        y_tr, y_va = data["y"]["train"], data["y"]["val"].numpy().astype(int)
        gen = torch.Generator().manual_seed(cfg.seed)
        loader = DataLoader(TensorDataset(*train_x, y_tr), batch_size=cfg.batch_size, shuffle=True, generator=gen)
        logger.info(f"model {name} | pipeline {pipe} | params {n_params} | vocab {len(vocab)} | train {len(y_tr)} | "
                    f"val {len(y_va)} | steps/epoch {len(loader)}")
        logger.debug(f"architecture {model}")
        reset_peak_mem()
        history, best, bad_epochs, step, examples, compute_s = [], -1.0, 0, 0, 0, 0.0
        t_start = time.perf_counter()
        for epoch in range(1, cfg.max_epochs + 1):
            model.train()
            ep_loss, ep_n = 0.0, 0
            for *xs, y in loader:
                ts = time.perf_counter()
                xs, y = [x.to(DEVICE) for x in xs], y.to(DEVICE)
                loss = F.binary_cross_entropy_with_logits(model(*xs), y)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                lv = loss.item(); sync(DEVICE)
                dt = time.perf_counter() - ts
                step += 1; examples += len(y); compute_s += dt; ep_loss += lv * len(y); ep_n += len(y)
                msg = f"step {step} epoch {epoch} loss {lv:.4f} step_s {dt:.4f} peak_mem_gb {peak_mem_gb():.3f}"
                (logger.info if step == 1 or step % cfg.log_every == 0 else logger.debug)(msg)
            p_va = predict(model, val_x)
            val_f1 = f1_score(y_va, (p_va >= 0.5).astype(int), average="macro")
            val_acc = accuracy_score(y_va, (p_va >= 0.5).astype(int))
            history.append({"epoch": epoch, "train_loss": ep_loss / ep_n, "val_f1_macro": val_f1, "val_accuracy": val_acc,
                            "elapsed_seconds": time.perf_counter() - t_start})
            logger.info(f"epoch {epoch}/{cfg.max_epochs} train_loss {ep_loss / ep_n:.4f} val_f1_macro {val_f1:.4f} "
                        f"val_accuracy {val_acc:.4f}")
            if val_f1 > best:
                best, bad_epochs = val_f1, 0
                torch.save({"model": model.state_dict(), "name": name, "pipeline": pipe, "vocab_size": len(vocab),
                            "epoch": epoch, "val_f1_macro": val_f1, "config": dict(cfg), "config_sha256": CFG_HASH}, ckpt)
                logger.info(f"new best val_f1_macro {val_f1:.4f} -> {rel(ckpt)}")
            else:
                bad_epochs += 1
                if bad_epochs >= cfg.early_stop_patience:
                    logger.info(f"early stop: no val macro-F1 gain for {bad_epochs} epochs")
                    break
        train_seconds = time.perf_counter() - t_start
        best_row = max(history, key=lambda h: h["val_f1_macro"])
        s = {"model": name, "pipeline": pipe, "param_count": n_params, "vocab_size": len(vocab),
             "best_epoch": best_row["epoch"], "epochs_run": len(history), "val_f1_macro": best_row["val_f1_macro"],
             "val_accuracy": best_row["val_accuracy"], "train_seconds": train_seconds,
             "examples_per_sec": examples / train_seconds, "compute_examples_per_sec": examples / compute_s,
             "peak_mem_gb": peak_mem_gb(), "device_name": device_name(DEVICE), "cpu_name": cpu_name(),
             "n_train": len(y_tr), "n_val": len(y_va), "history": history, "train_log": rel(log_path),
             "checkpoint": rel(ckpt), "config_sha256": CFG_HASH}
        summary_path.write_text(json.dumps(s, indent=2), encoding="utf-8")
        fig, ax1 = plt.subplots(figsize=(6, 3.5))
        ep = [h["epoch"] for h in history]
        ax1.plot(ep, [h["train_loss"] for h in history], "o-", label="train loss"); ax1.set_xlabel("epoch")
        ax1.set_ylabel("train BCE"); ax2 = ax1.twinx()
        ax2.plot(ep, [h["val_f1_macro"] for h in history], "s-", c="tab:orange", label="val macro-F1")
        ax2.set_ylabel("val macro-F1"); ax1.set_title(f"{name}: training curve"); ax1.grid(alpha=0.3)
        fig.legend(loc="center right"); fig.tight_layout(); fig.savefig(OUTPUTS / f"{name}_training_curve.png", dpi=150)
        plt.close(fig)
        logger.info("summary " + json.dumps({k: v for k, v in s.items() if k != "history"}))
        return s
    except BaseException:
        logger.exception("training stopped with an error")
        raise
    finally:
        close_run_log(logger)
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


summaries = {name: train_model(name) for name in cfg.models + (["meanpool_ablation"] if cfg.ablation else [])}
display(pd.DataFrame({k: {c: v[c] for c in ("param_count", "best_epoch", "epochs_run", "val_f1_macro", "val_accuracy",
                                            "train_seconds", "examples_per_sec", "peak_mem_gb", "device_name")}
                      for k, v in summaries.items()}).T)

## 13. Evaluate on the shared test split (brief 2.2.3, every metric)
- **`metrics_report.csv`** has exactly three rows, one per model.
- **The ablation** goes to `ablation_report.csv`, compared on validation macro-F1.
- **Files written:** a confusion-matrix and a reliability-diagram PNG per model, plus the test probabilities, saved for traceability.

In [ ]:
def load_model(name: str):
    ck = torch.load(CKPT_DIR / f"{name}_best.pt", map_location=DEVICE, weights_only=False)
    m = build_model(name, ck["vocab_size"]).to(DEVICE)
    m.load_state_dict(ck["model"])
    return m.eval(), ck


def plot_cm_and_reliability(name, y, p):
    cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 3.6))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["negative", "positive"]); ax.set_yticks([0, 1], ["negative", "positive"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{name}: confusion matrix (test)")
    fig.tight_layout(); fig.savefig(OUTPUTS / f"{name}_confusion_matrix.png", dpi=150); plt.close(fig)
    rb = reliability_bins(y, p, cfg.ece_bins)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot([0.5, 1], [0.5, 1], "k--", lw=0.8, label="perfect calibration")
    ax.plot(rb.confidence, rb.accuracy, "o-", label="model")
    ax.set_xlabel("confidence max(p, 1-p)"); ax.set_ylabel("accuracy"); ax.set_xlim(0.5, 1); ax.set_ylim(0, 1.02)
    ax.set_title(f"{name}: reliability (ECE {expected_calibration_error(y, p, cfg.ece_bins):.4f})")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(OUTPUTS / f"{name}_reliability.png", dpi=150)
    plt.close(fig)


def evaluate_all() -> pd.DataFrame:
    logger, log_path = open_run_log("evaluate")
    try:
        y = data["y"]["test"].numpy().astype(int)
        probs, rows = {}, []
        for name in cfg.models:
            m, ck = load_model(name)
            probs[name] = predict(m, model_inputs(name, "test"))
            np.save(OUTPUTS / f"{name}_test_probs.npy", probs[name])
            del m
        base_pred = (probs[cfg.baseline] >= 0.5).astype(int)
        for name in cfg.models:
            p, s = probs[name], summaries[name]
            pred = (p >= 0.5).astype(int)
            r = {"model": name, **full_report(y, p), **bootstrap_ci(y, pred, cfg.bootstrap_n, cfg.bootstrap_seed)}
            if name == cfg.baseline:
                r["mcnemar_stat_vs_baseline"], r["mcnemar_p_vs_baseline"] = float("nan"), float("nan")
            else:
                r["mcnemar_stat_vs_baseline"], r["mcnemar_p_vs_baseline"] = mcnemar_test(y, base_pred, pred)
            r.update(slice_report(y, pred, data["masks"]))
            r.update({k: s[k] for k in ("param_count", "train_seconds", "examples_per_sec", "peak_mem_gb",
                                        "device_name", "cpu_name", "best_epoch", "epochs_run", "val_f1_macro",
                                        "val_accuracy", "n_train", "n_val")})
            r.update({"n_test": len(y), "checkpoint": s["checkpoint"], "config_sha256": CFG_HASH})
            rows.append(r)
            plot_cm_and_reliability(name, y, p)
            logger.info(f"{name} " + json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()}))
        df = pd.DataFrame(rows)
        df.to_csv(METRICS_CSV, index=False, encoding="utf-8")
        if cfg.ablation:
            ab = pd.DataFrame([{"run": n, "stopwords_removed": cfg.remove_stopwords if n == "meanpool" else False,
                                "stemming": cfg.stem if n == "meanpool" else False,
                                **{k: summaries[n][k] for k in ("val_f1_macro", "val_accuracy", "best_epoch", "vocab_size")}}
                               for n in ("meanpool", "meanpool_ablation")])
            ab.to_csv(ABLATION_CSV, index=False, encoding="utf-8")
            logger.info("ablation " + ab.to_json(orient="records"))
        logger.info(f"wrote {rel(METRICS_CSV)} ({len(df)} rows)")
        data["probs"], data["eval_log"] = probs, rel(log_path)
        return df
    except BaseException:
        logger.exception("evaluation stopped with an error")
        raise
    finally:
        close_run_log(logger)


metrics = evaluate_all()
metrics.set_index("model").T

## 14. The 20-error review candidates (brief 2.2.4)
- **Which model:** the best by **validation** macro-F1. Choosing by test score would be selecting on the test set.
- **Which errors:** 20 test errors, picked by the fixed rules: 5 confident false positives, 5 confident false negatives, 5 near-threshold errors (0.45–0.55,
  falling back to the closest to 0.5), and 5 from that model's worst slice by macro-F1.
- **Output:** `outputs/error_review_candidates.md`. Copy it into `failure_analysis.md`, then assign each an error type and one testable fix
  **yourself**.

In [ ]:
best_name = max(cfg.models, key=lambda n: summaries[n]["val_f1_macro"])
row = metrics.set_index("model").loc[best_name]
worst_slice = min(data["masks"], key=lambda s: row[f"slice_{s}_f1"] if row[f"slice_{s}_n"] else float("inf"))
y_test = data["y"]["test"].numpy().astype(int)
p_best = data["probs"][best_name]
chosen, in_band = select_errors(y_test, p_best, data["masks"], worst_slice, tuple(cfg.near_band))

lines = [f"# Task 2 error review candidates: {best_name} (best validation macro-F1 {summaries[best_name]['val_f1_macro']:.4f})", "",
         f"Checkpoint `{summaries[best_name]['checkpoint']}`. Test rows index the official test.csv (0-based). "
         f"Worst slice by test macro-F1: **{worst_slice}** ({row[f'slice_{worst_slice}_f1']:.4f}). "
         f"Near-threshold errors inside [{cfg.near_band[0]}, {cfg.near_band[1]}]: **{in_band}** "
         f"({'all 5 from the band' if in_band >= 5 else 'the rest are the errors closest to 0.5'}).", ""]
for i, (cat, r) in enumerate(chosen, 1):
    flags = ", ".join(s for s, m in data["masks"].items() if m[r]) or "none"
    text = unescape(data["texts"]["test"][r]).replace("\n", " ")
    lines += [f"## {i}. {cat}", f"- test row: {r} | true: {'positive' if y_test[r] else 'negative'} | "
              f"p(positive) = {p_best[r]:.4f} | slices: {flags}",
              f"- review: {text[:1200]}{' ...' if len(text) > 1200 else ''}",
              "- error type: _(you)_", "- testable fix: _(you)_", ""]
ERRORS_MD.write_text("\n".join(lines), encoding="utf-8")
print(f"{len(chosen)} errors from {best_name} -> {rel(ERRORS_MD)}")
if len(chosen) < 20:
    print(f"WARNING: only {len(chosen)} qualifying test errors; the brief asks for 20. Say so in failure_analysis.md.")
display(pd.DataFrame([{"#": i, "category": c, "test_row": r, "true": int(y_test[r]), "p_pos": round(float(p_best[r]), 4),
                       "review (start)": unescape(data["texts"]["test"][r])[:120]} for i, (c, r) in enumerate(chosen, 1)]))

## 15. Results and manifest

In [ ]:
cols = ["accuracy", "f1_macro", "mcc", "roc_auc", "pr_auc", "brier", "ece", "acc_ci_lo", "acc_ci_hi",
        "f1_macro_ci_lo", "f1_macro_ci_hi", "mcc_ci_lo", "mcc_ci_hi", "mcnemar_stat_vs_baseline", "mcnemar_p_vs_baseline",
        "param_count", "train_seconds", "examples_per_sec", "peak_mem_gb", "device_name"]
display(metrics.set_index("model")[cols].round(4))
display(metrics.set_index("model")[[c for c in metrics.columns if c.startswith("slice_")]].round(4))
if cfg.ablation:
    display(pd.read_csv(ABLATION_CSV))
for name in cfg.models:
    display(Markdown(f"**{name}**"))
    for kind in ("training_curve", "confusion_matrix", "reliability"):
        display(Image(OUTPUTS / f"{name}_{kind}.png", width=380))

In [ ]:
all_runs = cfg.models + (["meanpool_ablation"] if cfg.ablation else [])
manifest = {
    "member": cfg.member, "task": "task2", "run_name": cfg.run_name, "config_sha256": CFG_HASH,
    "notebook": "task2_sentiment/sneha/src/task2_sentiment_sneha.ipynb", "git_sha": git_sha(), "seed": cfg.seed,
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": platform.python_version(), "platform": platform.platform(), "torch": torch.__version__,
    "cuda": torch.version.cuda, "device": device_name(DEVICE), "cpu": cpu_name(), "packages": installed_packages(),
    "data": {"train_csv_sha256": DATA_SHA["train.csv"], "test_csv_sha256": DATA_SHA["test.csv"],
             "split_files_sha256": SPLIT_SHA, "rows_after_cleaning": {k: len(v) for k, v in data["texts"].items()}},
    "checkpoints": {n: {"path": summaries[n]["checkpoint"], "sha256": sha256_file(ROOT / summaries[n]["checkpoint"]),
                        "bytes": (ROOT / summaries[n]["checkpoint"]).stat().st_size} for n in all_runs},
    "metric_rows": {summaries[n]["checkpoint"]: {"csv": rel(METRICS_CSV), "row": i + 1} for i, n in enumerate(cfg.models)},
    "ablation": {"csv": rel(ABLATION_CSV), "checkpoint": summaries["meanpool_ablation"]["checkpoint"]} if cfg.ablation else None,
    "raw_logs": [data["log"]] + [summaries[n]["train_log"] for n in all_runs] + [data["eval_log"]],
    "evidence": {"error_review": rel(ERRORS_MD), "plots": sorted(rel(p) for p in OUTPUTS.glob("*.png")),
                 "test_probs": sorted(rel(p) for p in OUTPUTS.glob("*_test_probs.npy"))},
    "config": dict(cfg),
}
MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("manifest:", rel(MANIFEST))
print("device:", manifest["device"], "| cpu:", manifest["cpu"], "| torch", manifest["torch"], "| git", manifest["git_sha"][:12])
for n, c in manifest["checkpoints"].items():
    print(f"  {n:18s} {c['path']} | sha256 {c['sha256'][:16]}... | {c['bytes'] / 1e6:.1f} MB")
print("raw logs:", *manifest["raw_logs"], sep="\n  ")

## 16. Analysis
Written by me, not generated:
- **Architecture and embedding justification, the model comparison, and whether stopword removal and stemming helped:**
  [`../results.md`](../results.md). Cite `ablation_report.csv` for the last one.
- **The 20-error review,** with an error type and a testable fix for each: [`../failure_analysis.md`](../failure_analysis.md).